In [1]:
# ==================================================
# SmogNet
# 06_Alert_Generator
# Public Health Alert System
# ==================================================

import pandas as pd
import numpy as np

df=spark.sql("""

SELECT *
FROM Classification_output

""").toPandas()

print(df.shape)

df.head()

StatementMeta(, 8ddf6c38-838c-43c8-8756-388af7d94616, 3, Finished, Available, Finished, False)

(1090, 61)


,datetime,main_aqi,components_co,components_no,components_no2,components_o3,components_so2,components_pm2_5,components_pm10,components_nh3,...,zscore,scaled_robust,scaled_z,scaled_iso,anomaly_score,anomaly_flag,severity,risk_level,predicted_source,confidence
0,6/9/2024 16:00,5,8972.17,80.47,400.30,0.0,54.36,269.53,292.96,72.96,...,2.594648,0.226946,0.000052,0.540609,0.183787,1,Critical,Low,Crop Burning,0.65
1,6/9/2024 17:00,5,10147.09,89.41,416.76,0.0,58.17,310.69,337.80,81.06,...,2.621716,0.268308,0.000052,0.476030,0.209620,1,Critical,Low,Crop Burning,0.65
2,6/9/2024 18:00,5,11215.21,97.45,427.72,0.0,58.65,349.03,379.09,87.14,...,2.553254,0.306836,0.000051,0.469614,0.212185,1,Critical,Low,Crop Burning,0.65
3,6/9/2024 19:00,5,11428.83,99.24,411.27,0.0,53.88,365.13,395.83,88.16,...,2.298341,0.323014,0.000046,0.479783,0.208114,1,Critical,Low,Crop Burning,0.65
4,6/9/2024 20:00,5,10894.78,87.62,367.40,0.0,42.44,356.55,387.61,83.09,...,1.927454,0.314392,0.000039,0.505038,0.198008,1,Critical,Low,Crop Burning,0.65


In [2]:
conditions=[

df["anomaly_score"]<0.4,

df["anomaly_score"]<0.6,

df["anomaly_score"]<0.8

]

choices=[

"Low",
"Moderate",
"High"

]

df["severity"]=np.select(

conditions,
choices,
default="Critical"

)

df["severity"].value_counts()

StatementMeta(, 8ddf6c38-838c-43c8-8756-388af7d94616, 4, Finished, Available, Finished, False)

severity
Low         1066
Moderate      18
High           6
Name: count, dtype: int64

In [3]:
def generate_alert(row):

    city=row["city"]
    source=row["predicted_source"]
    severity=row["severity"]


    # Source-specific cause

    if source=="Crop Burning":

        cause=(
        "seasonal agricultural burning activity"
        )

        advice=(
        "Keep windows closed and wear masks outdoors."
        )


    elif source=="Vehicular":

        cause=(
        "heavy traffic and vehicle emissions"
        )

        advice=(
        "Avoid congested roads and limit outdoor exercise."
        )


    elif source=="Industrial":

        cause=(
        "industrial emissions in surrounding areas"
        )

        advice=(
        "Reduce outdoor exposure and use protective masks."
        )


    elif source=="Dust Storm":

        cause=(
        "high dust movement and airborne particles"
        )

        advice=(
        "Remain indoors and avoid unnecessary travel."
        )

    else:

        cause=(
        "multiple overlapping pollution sources"
        )

        advice=(
        "Sensitive groups should reduce outdoor exposure."
        )


    # Severity customization

    if severity=="Critical":

        severity_text=(
        "Pollution levels are critically elevated."
        )

    elif severity=="High":

        severity_text=(
        "Air quality conditions are becoming hazardous."
        )

    elif severity=="Moderate":

        severity_text=(
        "Moderate pollution levels have been detected."
        )

    else:

        severity_text=(
        "Minor pollution increases have been observed."
        )


    msg=f"""
Air quality in {city} has shown unusual changes likely caused by {cause}. 
{severity_text}
Children, elderly individuals and respiratory patients may experience health impacts.
{advice}
"""

    return " ".join(msg.split())

df[
"alert_message"
]=df.apply(

generate_alert,

axis=1

)

df[
[
"city",
"predicted_source",
"severity",
"alert_message"
]
].head()

StatementMeta(, 8ddf6c38-838c-43c8-8756-388af7d94616, 5, Finished, Available, Finished, False)

,city,predicted_source,severity,alert_message
0,Islamabad,Crop Burning,Low,Air quality in Islamabad has shown unusual cha...
1,Islamabad,Crop Burning,Low,Air quality in Islamabad has shown unusual cha...
2,Islamabad,Crop Burning,Low,Air quality in Islamabad has shown unusual cha...
3,Islamabad,Crop Burning,Low,Air quality in Islamabad has shown unusual cha...
4,Islamabad,Crop Burning,Low,Air quality in Islamabad has shown unusual cha...


In [4]:
sample_df=df.sample(
    10,
    random_state=42
)

sample_df[
[
"city",
"predicted_source",
"severity",
"alert_message"
]
]

for x in sample_df[
"alert_message"
]:

    print(x)

    print("\n")

StatementMeta(, 8ddf6c38-838c-43c8-8756-388af7d94616, 6, Finished, Available, Finished, False)

Air quality in Peshawar has shown unusual changes likely caused by multiple overlapping pollution sources. Minor pollution increases have been observed. Children, elderly individuals and respiratory patients may experience health impacts. Sensitive groups should reduce outdoor exposure.


Air quality in Peshawar has shown unusual changes likely caused by seasonal agricultural burning activity. Minor pollution increases have been observed. Children, elderly individuals and respiratory patients may experience health impacts. Keep windows closed and wear masks outdoors.


Air quality in Karachi has shown unusual changes likely caused by seasonal agricultural burning activity. Minor pollution increases have been observed. Children, elderly individuals and respiratory patients may experience health impacts. Keep windows closed and wear masks outdoors.


Air quality in Quetta has shown unusual changes likely caused by high dust movement and airborne particles. Minor pollution increases have 

In [5]:
print(
"Total Alerts:"
)

print(
len(df)
)

print(
"\nCities:"
)

print(
df[
"city"
].nunique()
)

print(
"\nSources:"
)

print(
df[
"predicted_source"
].value_counts()
)

StatementMeta(, 8ddf6c38-838c-43c8-8756-388af7d94616, 7, Finished, Available, Finished, False)

Total Alerts:
1090

Cities:
5

Sources:
predicted_source
Crop Burning     693
Mixed Sources    201
Industrial        88
Vehicular         63
Dust Storm        45
Name: count, dtype: int64


In [7]:
spark.createDataFrame(
df
).write.mode(
"overwrite"
).format(
"delta"
).saveAsTable(
"Alert_output"
)

print(
"Alert output saved"
)

StatementMeta(, 8ddf6c38-838c-43c8-8756-388af7d94616, 10, Finished, Available, Finished, False)

Alert output saved
